In [24]:
# 06_mlp_hard_negative_distillation.ipynb
# Approach 2B: hard-negative Weighted-Jaccard distillation.
#
# The first distillation run improved GT-vs-random separation but hurt retrieval.
# This version trains on the actual mistakes that matter:
#   hard negatives = current MLP cosine candidates that are not WJ ground-truth neighbors.
#
# Goal:
#   Keep the useful original MLP geometry, but push WJ positives above hard cosine false positives.


In [25]:
import gc
import os
import pickle
import random
import time
from pathlib import Path

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

THREADS = 32
QUERY_START_10K = 8000
QUERY_START_FULL = 187019

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class QuadtreeCompressorV1Fixed(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt = set(gt_lookup.get(qid, [])[:k])
        if not gt:
            continue
        total += len(gt & set(ids[:k])) / len(gt)
        count += 1
    return total / count if count else 0.0

def eval_recall(gt_lookup, nbrs, query_start_id, max_k):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in (10, 50, 100, 500) if k <= max_k}

def generate_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks)

def build_cosine_index(corpus_embs, ef_search=200):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": ef_search})
    return idx, build_s, idx_mb


In [26]:
# Configuration
# Start with 10k. The method can scale to full, but mining hard negatives is heavier.
dataset_name = "full"      # "10k" or "full"
device = torch.device("cuda:0")
seed = 123

# Hard-negative mining
hard_pool_k = 500          # cosine candidates to inspect for hard negatives
exclude_gt_top = 500       # do not use true WJ top-500 as negatives
positive_per_query = 5
hard_neg_per_positive = 2
max_queries = None         # set e.g. 1000 for a smoke test

# Conservative fine-tuning: original MLP is already strong.
last_layer_only = True
batch_size = 256
epochs = 8
lr = 5e-5
weight_decay = 1e-4
margin = 0.05
rank_weight = 1.0
preserve_weight = 0.25

# Retrieval eval: if this works, it should let smaller K match old larger K.
candidate_ks = [100, 200, 500]
rerank_batch_size = 4

out_path = "/tmp/results_hardneg_wjdistill.pkl"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [27]:
# Full dataset configuration
# Run this cell instead of the default Configuration cell when you want the full 233k experiment.
# Suggested workflow:
#   1. Restart kernel
#   2. Run imports/model definitions
#   3. Run this full config cell
#   4. Continue with Load data/checkpoint onward

# Full run: direct baseline comparison.
dataset_name = "full"
device = torch.device("cuda:0")
seed = 123

# Hard-negative mining. This creates roughly:
#   valid_queries * positive_per_query * hard_neg_per_positive triplets
# Full GT has about 44k valid queries, so defaults make about 440k triplets.
hard_pool_k = 500
exclude_gt_top = 500
positive_per_query = 5
hard_neg_per_positive = 2
max_queries = None

# Conservative fine-tuning. Keep last_layer_only=True for the first full run.
last_layer_only = True
batch_size = 256
epochs = 6
lr = 5e-5
weight_decay = 1e-4
margin = 0.05
rank_weight = 1.0
preserve_weight = 0.25

# Retrieval eval. K=500 is the strongest direct comparison point.
# K=1000 is useful but substantially more GPU rerank memory/time.
candidate_ks = [1000, 2000, 5000]
rerank_batch_size = 8

out_path = "/tmp/results_hardneg_wjdistill.pkl"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("Configured FULL hard-negative WJ distillation run")
print(f"candidate_ks={candidate_ks} | rerank_batch_size={rerank_batch_size} | epochs={epochs}")


Configured FULL hard-negative WJ distillation run
candidate_ks=[1000, 2000, 5000] | rerank_batch_size=8 | epochs=6


In [28]:
# Load data/checkpoint
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
    base_ckpt = "/tmp/best_compressor_v1_clean.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_10k.pt"
    model_cls = QuadtreeCompressorV1
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
    base_ckpt = "/tmp/best_compressor_full_fixed.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_full.pt"
    model_cls = QuadtreeCompressorV1Fixed
else:
    raise ValueError(dataset_name)

corpus_qt = qt[:query_start]
query_qt = qt[query_start:]
corpus_size = len(corpus_qt)
query_ids = [qid for qid in sorted(gt) if query_start <= qid < len(qt)]
if max_queries is not None:
    query_ids = query_ids[:max_queries]

print(f"dataset={dataset_name}")
print(f"qt={qt.shape} | corpus={corpus_qt.shape} | queries={query_qt.shape}")
print(f"teacher queries used={len(query_ids)}")
print(f"base checkpoint={base_ckpt}")
print(f"hard-neg checkpoint={hard_ckpt}")


dataset=full
qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)
teacher queries used=44666
base checkpoint=/tmp/best_compressor_full_fixed.pt
hard-neg checkpoint=/tmp/best_compressor_hardneg_wjdistill_full.pt


In [29]:
# Mine hard negatives using the current/base MLP cosine index
base_model = model_cls(qt.shape[1], out_dim=512).to(device)
base_model.load_state_dict(torch.load(base_ckpt, weights_only=True, map_location=device))
base_model.eval()

base_embs = generate_embeddings(base_model, qt, device)
base_corpus_embs = base_embs[:query_start]
base_query_embs = base_embs[query_start:]
base_idx, base_build_s, base_idx_mb = build_cosine_index(base_corpus_embs)

local_query_indices = [qid - query_start for qid in query_ids]
query_emb_subset = base_query_embs[local_query_indices]
print(f"Mining top-{hard_pool_k} cosine candidates for {len(query_emb_subset)} queries...")
t0 = time.time()
mined_nbrs = base_idx.knnQueryBatch(query_emb_subset, k=hard_pool_k, num_threads=THREADS)
print(f"Mining time: {time.time() - t0:.2f}s")


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 717489.95it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Mining top-500 cosine candidates for 44666 queries...
Mining time: 17.59s


In [30]:
# Build hard-negative training triplets
# Each triplet stores query id, WJ-positive corpus id, hard-negative corpus id.
triplets = []
missed_hard = 0
for qid, (cand_ids, _) in tqdm(zip(query_ids, mined_nbrs), total=len(query_ids), desc="Building hard triplets"):
    positives = [pid for pid in gt.get(qid, []) if 0 <= pid < corpus_size]
    if not positives:
        continue

    pos_train = positives[:positive_per_query]
    exclude = set(positives[:exclude_gt_top])
    hard_negs = [int(cid) for cid in cand_ids if int(cid) not in exclude]

    if not hard_negs:
        missed_hard += 1
        continue

    for pos_id in pos_train:
        for j in range(hard_neg_per_positive):
            neg_id = hard_negs[min(j, len(hard_negs) - 1)]
            triplets.append((qid, pos_id, neg_id))

random.shuffle(triplets)
val_frac = 0.1
val_n = max(1, int(len(triplets) * val_frac))
val_triplets = triplets[:val_n]
train_triplets = triplets[val_n:]

print(f"hard-neg triplets total={len(triplets):,} | train={len(train_triplets):,} | val={len(val_triplets):,}")
print(f"queries with no usable hard negatives={missed_hard}")
print("sample:", triplets[0] if triplets else None)


Building hard triplets: 100%|██████████| 44666/44666 [00:15<00:00, 2947.79it/s]


hard-neg triplets total=441,082 | train=396,974 | val=44,108
queries with no usable hard negatives=0
sample: (218610, 167702, 588)


In [31]:
class HardNegTripletDataset(Dataset):
    def __init__(self, qt, corpus_qt, triplets):
        self.qt = qt
        self.corpus_qt = corpus_qt
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        qid, pos_id, neg_id = self.triplets[idx]
        return (
            torch.from_numpy(self.qt[qid]).float(),
            torch.from_numpy(self.corpus_qt[pos_id]).float(),
            torch.from_numpy(self.corpus_qt[neg_id]).float(),
        )

train_loader = DataLoader(HardNegTripletDataset(qt, corpus_qt, train_triplets),
                          batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(HardNegTripletDataset(qt, corpus_qt, val_triplets),
                        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

def set_trainable_scope(model, last_layer_only=True):
    if not last_layer_only:
        for p in model.parameters():
            p.requires_grad = True
        return
    for p in model.parameters():
        p.requires_grad = False
    # Sequential layout: Linear, BN, ReLU, Linear, BN, ReLU, Linear, BN
    for layer_idx in (6, 7):
        for p in model.net[layer_idx].parameters():
            p.requires_grad = True

def hardneg_loss(model, teacher_model, q, pos, neg):
    zq = F.normalize(model(q), dim=1)
    zp = F.normalize(model(pos), dim=1)
    zn = F.normalize(model(neg), dim=1)

    sim_pos = F.cosine_similarity(zq, zp)
    sim_neg = F.cosine_similarity(zq, zn)
    rank = F.relu(margin - sim_pos + sim_neg).mean()

    with torch.no_grad():
        tq = F.normalize(teacher_model(q), dim=1)
        tp = F.normalize(teacher_model(pos), dim=1)
        tn = F.normalize(teacher_model(neg), dim=1)

    preserve = (
        F.mse_loss(zq, tq) +
        F.mse_loss(zp, tp) +
        F.mse_loss(zn, tn)
    ) / 3.0
    loss = rank_weight * rank + preserve_weight * preserve
    return loss, rank.detach(), preserve.detach(), sim_pos.detach().mean(), sim_neg.detach().mean()


In [32]:
# Fine-tune on hard negatives while preserving base geometry
teacher_model = model_cls(qt.shape[1], out_dim=512).to(device)
teacher_model.load_state_dict(torch.load(base_ckpt, weights_only=True, map_location=device))
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

model = model_cls(qt.shape[1], out_dim=512).to(device)
model.load_state_dict(torch.load(base_ckpt, weights_only=True, map_location=device))
set_trainable_scope(model, last_layer_only=last_layer_only)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_val = float("inf")
history = []
for epoch in range(1, epochs + 1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs} train", leave=False)
    for q, pos, neg in pbar:
        q = q.to(device, non_blocking=True)
        pos = pos.to(device, non_blocking=True)
        neg = neg.to(device, non_blocking=True)

        loss, rank, preserve, sim_pos, sim_neg = hardneg_loss(model, teacher_model, q, pos, neg)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        train_losses.append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=f"{train_losses[-1]:.4f}", rank=f"{float(rank):.4f}", preserve=f"{float(preserve):.4f}")

    model.eval()
    val_losses = []
    with torch.no_grad():
        for q, pos, neg in val_loader:
            q = q.to(device, non_blocking=True)
            pos = pos.to(device, non_blocking=True)
            neg = neg.to(device, non_blocking=True)
            loss, rank, preserve, sim_pos, sim_neg = hardneg_loss(model, teacher_model, q, pos, neg)
            val_losses.append(float(loss.detach().cpu()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), hard_ckpt)

    print(f"Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f} | best={best_val:.5f} | lr={scheduler.get_last_lr()[0]:.2e}")

print(f"Done. Best checkpoint: {hard_ckpt}")


Trainable params: 525,312 / 79,358,976


Epoch 01 | train=0.00752 | val=0.01415 | best=0.01415 | lr=4.67e-05


Epoch 02 | train=0.00709 | val=0.02612 | best=0.01415 | lr=3.75e-05


Epoch 03 | train=0.00672 | val=0.01011 | best=0.01011 | lr=2.50e-05


Epoch 04 | train=0.00651 | val=0.01845 | best=0.01011 | lr=1.25e-05


Epoch 05 | train=0.00638 | val=0.01189 | best=0.01011 | lr=3.35e-06


Epoch 06 | train=0.00641 | val=0.01025 | best=0.01011 | lr=0.00e+00
Done. Best checkpoint: /tmp/best_compressor_hardneg_wjdistill_full.pt


In [33]:
# Quality check: hard positive vs mined hard negative cosine gap
# This is more useful than GT-vs-random for candidate generation.
def hard_quality(model, triplets, n=1000):
    model.eval()
    sample = triplets[:min(n, len(triplets))]
    pos_sims, neg_sims = [], []
    with torch.no_grad():
        for qid, pos_id, neg_id in tqdm(sample, desc="Hard quality"):
            q = torch.tensor(qt[qid], dtype=torch.float32, device=device).unsqueeze(0)
            p = torch.tensor(corpus_qt[pos_id], dtype=torch.float32, device=device).unsqueeze(0)
            nvec = torch.tensor(corpus_qt[neg_id], dtype=torch.float32, device=device).unsqueeze(0)
            zq = F.normalize(model(q), dim=1)
            zp = F.normalize(model(p), dim=1)
            zn = F.normalize(model(nvec), dim=1)
            pos_sims.append(F.cosine_similarity(zq, zp).item())
            neg_sims.append(F.cosine_similarity(zq, zn).item())
    return float(np.mean(pos_sims)), float(np.mean(neg_sims)), float(np.mean(pos_sims) - np.mean(neg_sims))

hard_model = model_cls(qt.shape[1], out_dim=512).to(device)
hard_model.load_state_dict(torch.load(hard_ckpt, weights_only=True, map_location=device))

base_hq = hard_quality(base_model, triplets)
hard_hq = hard_quality(hard_model, triplets)
print(f"Base     hard-pos={base_hq[0]:.4f} | hard-neg={base_hq[1]:.4f} | gap={base_hq[2]:.4f}")
print(f"HardDist hard-pos={hard_hq[0]:.4f} | hard-neg={hard_hq[1]:.4f} | gap={hard_hq[2]:.4f}")


Hard quality: 100%|██████████| 1000/1000 [00:01<00:00, 743.13it/s]

Base     hard-pos=0.9170 | hard-neg=0.8213 | gap=0.0957
HardDist hard-pos=0.9248 | hard-neg=0.8347 | gap=0.0901


In [34]:
# Two-stage evaluation with GPU exact WJ rerank
def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[absolute_i] for absolute_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = (ids[row].tolist(), [])
    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return reranked

def evaluate_two_stage(eval_model, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    embs = generate_embeddings(eval_model, qt, device)
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    vec_mb = corpus_embs.nbytes / 1024**2
    idx, build_s, idx_mb = build_cosine_index(corpus_embs)
    corpus_sums = corpus_qt.sum(axis=1)
    results = {}
    for k in candidate_ks:
        t0 = time.time()
        nbrs_raw = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0
        t0 = time.time()
        nbrs_rr = rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, rerank_batch_size)
        rerank_s = time.time() - t0
        qps = len(query_embs) / (hnsw_s + rerank_s)
        rec = eval_recall(gt, nbrs_rr, query_start, max_k=k)
        results[f"k{k}_wj_rerank"] = {**rec, "candidate_k": k, "qps": qps,
                                      "hnsw_s": hnsw_s, "rerank_s": rerank_s,
                                      "build_s": build_s, "vec_mb": vec_mb, "idx_mb": idx_mb}
        print(f"K={k} | QPS={qps:.1f} | HNSW={hnsw_s:.2f}s | WJ={rerank_s:.2f}s")
        for kk, rr in rec.items():
            print(f"  R@{kk:<4} = {rr:.4f}")
    return results

base_eval_results = evaluate_two_stage(base_model, "Base MLP + cosine candidates + GPU WJ rerank")
hard_eval_results = evaluate_two_stage(hard_model, "Hard-negative distilled MLP + cosine candidates + GPU WJ rerank")



Base MLP + cosine candidates + GPU WJ rerank


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 590418.85it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
GPU WJ rerank: 100%|██████████| 5845/5845 [00:20<00:00, 286.93it/s]****


K=1000 | QPS=986.3 | HNSW=25.93s | WJ=21.47s
  R@10   = 0.9927
  R@50   = 0.9952
  R@100  = 0.9949
  R@500  = 0.8983


GPU WJ rerank: 100%|██████████| 5845/5845 [00:39<00:00, 149.41it/s]


K=2000 | QPS=583.9 | HNSW=39.30s | WJ=40.77s
  R@10   = 0.9927
  R@50   = 0.9952
  R@100  = 0.9951
  R@500  = 0.9166


...

K=1000 | QPS=1000.1 | HNSW=23.92s | WJ=22.82s
  R@10   = 0.9926
  R@50   = 0.9951
  R@100  = 0.9948
  R@500  = 0.9034


GPU WJ rerank: 100%|██████████| 5845/5845 [00:38<00:00, 153.24it/s]


K=2000 | QPS=643.4 | HNSW=33.09s | WJ=39.58s
  R@10   = 0.9926
  R@50   = 0.9952
  R@100  = 0.9950
  R@500  = 0.9183


GPU WJ rerank: 100%|██████████| 5845/5845 [00:39<00:00, 147.17it/s]


K=5000 | QPS=631.4 | HNSW=32.44s | WJ=41.60s
  R@10   = 0.9926
  R@50   = 0.9952
  R@100  = 0.9950
  R@500  = 0.9184


In [35]:
# Save hard-negative distillation results and print compact comparison
run_key = time.strftime(f"{dataset_name}_hardneg_%Y%m%d_%H%M%S")
record = {
    "config": {
        "dataset_name": dataset_name,
        "hard_pool_k": hard_pool_k,
        "exclude_gt_top": exclude_gt_top,
        "positive_per_query": positive_per_query,
        "hard_neg_per_positive": hard_neg_per_positive,
        "last_layer_only": last_layer_only,
        "epochs": epochs,
        "lr": lr,
        "margin": margin,
        "rank_weight": rank_weight,
        "preserve_weight": preserve_weight,
        "candidate_ks": candidate_ks,
    },
    "history": history,
    "base_quality": base_hq,
    "hard_quality": hard_hq,
    "base_eval": base_eval_results,
    "hard_eval": hard_eval_results,
}

try:
    with open(out_path, "rb") as f:
        saved = pickle.load(f)
except FileNotFoundError:
    saved = {"runs": {}}
if "runs" not in saved:
    saved = {"runs": {"legacy": saved}}
saved["runs"][run_key] = record
with open(out_path, "wb") as f:
    pickle.dump(saved, f)
print(f"Saved run {run_key} to {out_path}")

print("\n" + "=" * 96)
print("BASE VS HARD-NEGATIVE DISTILLED TWO-STAGE RESULTS")
print(f"{'=' * 96}")
print(f"{'Model':<12} {'Method':<18} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'QPS':>9}")
print("-" * 96)
for label, results in [("Base", base_eval_results), ("HardDist", hard_eval_results)]:
    for method, res in results.items():
        def fmt(k):
            return f"{res[k]:>7.4f}" if k in res else f"{'-':>7}"
        print(f"{label:<12} {method:<18} {fmt(10)} {fmt(50)} {fmt(100)} {fmt(500)} {res['qps']:>9.1f}")


Saved run full_hardneg_20260427_122346 to /tmp/results_hardneg_wjdistill.pkl

BASE VS HARD-NEGATIVE DISTILLED TWO-STAGE RESULTS
Model        Method                R@10    R@50   R@100   R@500       QPS
------------------------------------------------------------------------------------------------
Base         k1000_wj_rerank     0.9927  0.9952  0.9949  0.8983     986.3
Base         k2000_wj_rerank     0.9927  0.9952  0.9951  0.9166     583.9
Base         k5000_wj_rerank     0.9927  0.9953  0.9951  0.9167     543.6
HardDist     k1000_wj_rerank     0.9926  0.9951  0.9948  0.9034    1000.1
HardDist     k2000_wj_rerank     0.9926  0.9952  0.9950  0.9183     643.4
HardDist     k5000_wj_rerank     0.9926  0.9952  0.9950  0.9184     631.4
